# 00. Setup & DB 연결 검증

**ScenarioDB — Jupyter 작업 환경 초기화**

이 노트북은 다음을 수행합니다:
1. DB 연결 검증 (PostgreSQL ping)
2. 9개 핵심 테이블 row count 표시
3. 각 테이블 sample row 1개 확인
4. 이후 노트북에서 쓸 공통 import 정리

---

## 0. 공통 Import

이후 모든 노트북에서 이 셀을 복사해 사용.

In [ ]:
import sys
from pathlib import Path

# repo root를 sys.path에 추가 (utils import 보장)
REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# 표준 라이브러리
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tabulate import tabulate
from IPython.display import display, HTML

# 프로젝트 유틸
from demo.notebooks.utils.db_connection import get_engine, query_df, ping
from demo.notebooks.utils.plot_theme import apply_theme, FEASIBILITY_COLORS, GATE_COLORS
from demo.notebooks.utils.query_helpers import (
    table_counts, sample_rows,
    list_ips, list_sw_profiles,
    list_variants, evidence_summary,
    compare_sw_versions, review_summary,
)

# 테마 적용
apply_theme()
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_columns', 20)

ENGINE = get_engine()
print('✓ 공통 import 완료')

## 1. DB 연결 검증

In [ ]:
version = ping(ENGINE)
print(f'PostgreSQL 연결 성공\n{version}')

## 2. 테이블 Row Count

In [ ]:
df_counts = table_counts(ENGINE)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(
    df_counts['table'], df_counts['count'],
    color=['#3498DB' if c > 0 else '#DEE2E6' for c in df_counts['count']]
)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_xlabel('Row Count')
ax.set_title('ScenarioDB — 테이블별 레코드 수', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(df_counts.style.bar(subset=['count'], color='#AED6F1').format({'count': '{:,}'}))

## 3. 테이블별 Sample Row

In [ ]:
TABLES = [
    'soc_platforms', 'ip_catalog', 'sw_profiles', 'sw_components',
    'scenarios', 'scenario_variants',
    'evidence', 'waivers', 'reviews',
]

for table in TABLES:
    df = sample_rows(table, n=1, engine=ENGINE)
    if df.empty:
        print(f'\n### {table} — (empty)')
        continue
    display(HTML(f'<h4 style="margin-bottom:4px">▶ {table}</h4>'))
    display(df.T.rename(columns={0: 'value'}))

## 4. LLC Thrashing Before / After 빠른 확인

Phase A 데모 스토리의 핵심 — sw-vendor-v1.2.3 vs v1.3.0 KPI 비교.

In [ ]:
df_cmp = compare_sw_versions(
    scenario_ref='uc-camera-recording',
    variant_ref='UHD60-HDR10-H265',
    engine=ENGINE,
)

display(HTML('<h4>KPI 비교: sw-vendor-v1.2.3 vs v1.3.0</h4>'))
display(df_cmp[['sw_version_hint','overall_feasibility',
                'total_power_mw','peak_power_mw',
                'avg_ddr_bw_gbps','frame_latency_ms']].style
    .background_gradient(subset=['total_power_mw','peak_power_mw'], cmap='RdYlGn_r')
    .map(
        lambda v: f'background-color: {FEASIBILITY_COLORS.get(v, "")}22; font-weight:bold',
        subset=['overall_feasibility']
    )
)

# KPI bar chart
kpi_cols = ['total_power_mw', 'peak_power_mw', 'avg_ddr_bw_gbps', 'frame_latency_ms']
df_melt = df_cmp[['sw_version_hint'] + kpi_cols].melt(
    id_vars='sw_version_hint', var_name='kpi', value_name='value'
)

fig = px.bar(
    df_melt, x='kpi', y='value', color='sw_version_hint',
    barmode='group',
    title='KPI Before/After — sw-vendor-v1.2.3 vs v1.3.0',
    color_discrete_map={
        'sw-vendor-v1.2.3': FEASIBILITY_COLORS['exploration_only'],
        'sw-vendor-v1.3.0': FEASIBILITY_COLORS['production_ready'],
    },
    labels={'sw_version_hint': 'SW Version', 'value': 'Value', 'kpi': 'KPI'},
)
fig.update_layout(legend_title_text='SW Baseline')
fig.show()

## 5. Review Gate 상태 확인

In [ ]:
df_reviews = review_summary(ENGINE)

def gate_badge(v):
    color = {'PASS': '#27AE60', 'WARN': '#E67E22', 'BLOCK': '#C0392B'}.get(v, '#95A5A6')
    return f'background-color:{color}22; color:{color}; font-weight:bold'

display(
    df_reviews[['id','gate_result','decision','status','waiver_ref','approver_claim','claim_at']]
    .style.map(gate_badge, subset=['gate_result'])
)

---

## 공통 Import 템플릿 (복사용)

다음 노트북 작성 시 Cell 0에 붙여넣기:

```python
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

from demo.notebooks.utils.db_connection import get_engine, query_df, ping
from demo.notebooks.utils.plot_theme import apply_theme, FEASIBILITY_COLORS, GATE_COLORS
from demo.notebooks.utils.query_helpers import *

apply_theme()
ENGINE = get_engine()
```